In [1]:
import mbuild as mb
from mbuild import Polymer

from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

gaff = ForceField("../files/gaff.xml")
oplsa = ForceField("oplsaa")
spcfw = ForceField("../files/spcfw.xml")

/Users/cj4006/miniforge3/envs/mosdef/lib/python3.12/site-packages/forcefield_utilities/foyer_xml.py:360: UserWarning: Tag <cyfunction Comment at 0x11f0947a0> not understood skipping
  warnings.warn(


# Building polymers in mBuild
- Tagging bonding sites with SMILES strings:
    - This implementation is inspired by polymer syntax approaches such as BIGSmiles, CGSmiles and more.
    - It is not meant to be a full implementation of the above syntax approaches
    - You can put any tag value inside the `{}` brackets and carry it over to the polymer builder

In [2]:
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build(n=4)
chain.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build(n=34)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Correct bonding graph, but bad configurations
- By using tagged SMILES strings, and the Polymer() class in mBuild, we can easily generate the correct polymer bonding topology
- However, we have no control over the chain configuraiton
- Try: Change `n` to a large number (40 - 50) in the thiophene example
- Challenges: Long straight chains create challenges in building polymer systems

In [4]:
from mbuild.path import Path, straight_line, lamellar, cyclic, hard_sphere_random_walk, lamellar, knot

In [5]:
line_path = straight_line(N=20, spacing=0.40)
line_path.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
lamellar_2D = lamellar(spacing=0.45, layer_length=5, layer_separation=0.8, num_layers=5)
lamellar_2D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
lamellar_3D = lamellar(spacing=0.45, layer_length=5, layer_separation=0.8, num_layers=5, stack_separation=1.2, num_stacks=4)
lamellar_3D.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
ring = cyclic(N=30, spacing=0.45)
ring.visualize(radius=0.40)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Coarse-grained chains
As we can see, the structures generated by all of these different "paths" are essentially coarse-grained polymers

Using mBuild polymer builder to back-map

In [9]:
ring = cyclic(N=30, spacing=0.30)

peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=ring, bond_head_tail=True, add_hydrogens=False)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
lamellar_2D = lamellar(spacing=0.40, layer_length=5, layer_separation=0.8, num_layers=5)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_2D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
lamellar_3D = lamellar(spacing=0.40, layer_length=5, layer_separation=0.8, num_layers=5, stack_separation=0.9, num_stacks=4)
monomer = mb.load("c1c{<}sc{>}c1", smiles=True)
chain = Polymer()
chain.add_monomer(monomer, head_tag=">", tail_tag="<", separation=0.145)
chain.build_from_path(lamellar_3D)
chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
random_walk_path = hard_sphere_random_walk(radius=0.30, bond_length=0.301, rw_angles=(1.5, 3.14), termination=25)
random_walk_path.visualize(radius=0.30)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=random_walk_path)
peg_chain.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Random Walks

In [39]:
from mbuild.path import Path
from mbuild.path.constraints import CuboidConstraint, SphereConstraint, CylinderConstraint
from mbuild.path.termination import NumSites, NumAttempts, WithinCoordinate
from mbuild.path.bias import TargetType, TargetDirection, AvoidType, AvoidDirection, TargetCoordinate
from mbuild.path.points import AnglesSampler

**Note: For the purpose of this example, the packing density is purposely kept low. If we have time, the next tutorial will go into details about how to pack dense, multi-chain systems.**

In [32]:
n_chains = 20
chain_lengths = 50

box = CuboidConstraint(Lx=8, Ly=8, Lz=8, pbc=(False, False, False))
path = Path()
seed = 12
for i in range(n_chains):
    random_walk = hard_sphere_random_walk(
        path=path,
        radius=0.35,
        volume_constraint=box,
        bond_length=0.351,
        seed=seed+i,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
    )

In [33]:
path.visualize(radius=0.35)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.